In [ ]:
using JuMP
using Gurobi
using Random
using Dualization
using Plots
using DataFrames
using CSV
import XLSX
import JSON

current_directory = @__DIR__
functions_directory = joinpath(current_directory, "functions")
data_dir = joinpath(current_directory, "data")
results_dir = joinpath(current_directory, "results")

# Include all the function files
# include(joinpath(functions_directory, "create_check_params.jl"))
# include(joinpath(functions_directory, "deterministic_equivalent.jl"))
# include(joinpath(functions_directory, "generate_cuts_from_dual.jl"))
include(joinpath(functions_directory, "load_model_starting_points.jl"))
include(joinpath(functions_directory, "initialize_parameters.jl"))
include(joinpath(functions_directory, "process_scenario_data.jl"))
# include(joinpath(functions_directory, "save_L_shaped_results.jl"))
include(joinpath(functions_directory, "select_random_scenarios.jl"))
include(joinpath(functions_directory, "create_vaccine_data.jl"))
# include(joinpath(functions_directory, "sub_problem.jl"))
# include(joinpath(functions_directory, "master_problem.jl"))

In [ ]:
unit = 1000
A, V, A_v, P, P_v, V_a, V_p, P_a, A_p, capacity_category, vaccine_category, antigen_category = create_vaccine_data()
T, T_initial, Δ, s_real, r, r_avg, r_producer_avg, g, h, l, f_profit, Γ, F_time_set, κ, L_lower_number, L_upper_number, delta, inf_penalty = initialize_parameters(data_dir, unit, 1, 10, 5, P, V, P_v, V_p, 1)

In [ ]:
m_segments = [0.05, 0.1, 0.2]
for m in eachindex(m_segments)
    println(m)
end

In [ ]:
T, T_initial, Δ, s_real, r, r_avg, r_producer_avg, g, h, l, f_profit, Γ, F_time_set, κ, L_lower_number, L_upper_number, delta, inf_penalty, zeta_vm, phi_vm_lower, phi_vm_upper, m_segments = initialize_parameters(
    data_dir, 1000, 1, 3, 3, 
    P, V, P_v, V_p, 1
)

In [ ]:
starting_points_vect_F, starting_points_vect_I, starting_points_vect_S = load_model_starting_points(data_dir, 1, unit)

In [ ]:
gurobi_solver = JuMP.optimizer_with_attributes(Gurobi.Optimizer, "FeasibilityTol" => 1e-4, "OutputFlag" => 0, "Presolve" => 0, "NumericFocus" => 1, "MIPGap" => 1e-3, "Threads" => 8) 
Masterproblem = JuMP.Model()
JuMP.set_optimizer(Masterproblem, gurobi_solver)

# @variable(Masterproblem, 0.0 <= F[a in A, (t, tau) in F_time_set] <= 1.0)
# @variable(Masterproblem, F[a in A, (t, tau) in F_time_set], Bin)
# @variable(Masterproblem, Q[v in V, p in P_v[v], (t, tau) in F_time_set, m in keys(m_segments)] >= 0)
# @variable(Masterproblem, Z[v in V, p in P_v[v], t in T, m in keys(m_segments)] >= 0)
@variable(Masterproblem, W[p in P, (t, tau) in F_time_set], Bin)
@variable(Masterproblem, L_1[p in P, t in T] >= 0)
@variable(Masterproblem, L_2[p in P, t in T] >= 0)
# open("Z_out.txt", "w") do io
#     println(io, Z)
# end

@objective(Masterproblem, Min, (1+1))


for p in P[1:1]
    for t in T
        for tau in T
            if (t,tau) in F_time_set
                @constraint(Masterproblem, sum(0) <=  1*((tau - t + 1)*s_real[p]) + sum(sum(s_real[p] * L_2[p,k] for k in 1:l) for l in t:tau))
            end
        end
    end
end


for p in P[1:1]
    for t in T
        for tau in T
            if (t,tau) in F_time_set
                @constraint(Masterproblem, sum(0) <=  1*((tau - t + 1)*s_real[p]) + sum((tau - t + 1)s_real[p] * L_1[p,k] for k in 1:t) + sum((tau - k + 1)s_real[p] * L_1[p,k] for k in t+1:tau))
            end
        end
    end
end


                    


In [ ]:
print(Masterproblem)

In [14]:
using JuMP, Gurobi

# Initialize the model with Gurobi solver
model = Model(Gurobi.Optimizer)

# Define integer variables
@variable(model, 0 <= x <= 1, Bin)
@variable(model, 0 <= y <= 10, Int)

# Objective function: Maximize x + 2y
@objective(model, Max, x + 2y)

# Constraints
@constraint(model, x + y <= 10)
@constraint(model, x - 2y >= -3)

# --- Step 1: Relax Integer Constraints ---
unset_binary(x)  # Relax x
unset_integer(y)  # Relax y

# Solve the LP relaxation
println("Solving relaxed problem (continuous)...")
optimize!(model)
println("Relaxed solution: x =", value(x), ", y =", value(y))

# --- Step 2: Restore Integer Constraints ---
set_binary(x)  # Convert x back to integer
set_integer(y)  # Convert y back to integer

# Solve the integer problem
println("Solving integer problem...")
optimize!(model)

# Display final integer solution
println("Integer solution: x =", value(x), ", y =", value(y))


Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-04
Solving relaxed problem (continuous)...
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0x651277b1
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [1e+00, 1e+01]
  RHS range        [3e+00, 1e+01]
Presolve removed 2 rows and 2 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.000000000e+00

User-callback calls 38, time in user-callback 0.00 sec
Re